**Fig. 3, step 4: score the bootstrap and from-scratch reconstructions against the ground truth (3D AP@50)**

In [ ]:
# Working directory: data/fig3/ under the repository root, where the Fig. 3 deposit folder is downloaded
# (decomposed_2D/, bootstrap_*_models/, domain-specialized_*/, benchmark_outputs*/, metrics_outputs/).
# The 11 ground-truth stacks are read from ../ground_truth/zebrafish/{development,held_out}/.
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # the notebook's own folder, for benchmarking_scripts/
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig3")
print("working directory:", os.getcwd())


# 3D Reconstruction Benchmark Metrics

This notebook evaluates reconstructed 3D segmentation masks produced by
multiple model pairs across multiple patches.

Each (patch × model_pair) is treated as one evaluation sample.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import tifffile as tif
from tqdm import tqdm
import os


In [ ]:
# Root directory containing all reconstruction outputs
RESULTS_ROOT = Path("benchmark_outputs")

# Ground truth directory

# where the metric table is written
OUT_DIR = Path("metrics_outputs")
OUT_DIR.mkdir(exist_ok=True)

In [ ]:
# ground-truth masks of the 11 stacks
data = [f'../ground_truth/zebrafish/development/{i}' for i in os.listdir('../ground_truth/zebrafish/development/') if 'mask' in i]
data_holdout = [f'../ground_truth/zebrafish/held_out/{i}' for i in os.listdir('../ground_truth/zebrafish/held_out/') if 'mask' in i]

gt_patches = data + data_holdout

gt_masks = {os.path.basename(i).split('_')[0]: Path(i) for i in gt_patches}
gt_masks

{'patch-5': PosixPath('data/patch-5_masks.tif'),
 'patch-7': PosixPath('data/patch-7_masks.tif'),
 'patch-8': PosixPath('data/patch-8_masks.tif'),
 'patch-4': PosixPath('data/patch-4_masks.tif'),
 'patch-2': PosixPath('data/patch-2_masks.tif'),
 'patch-3': PosixPath('data/patch-3_masks.tif'),
 'patch-1': PosixPath('data/patch-1_masks.tif'),
 'patch-14': PosixPath('data_holdout/patch-14_mask.tif'),
 'patch-9': PosixPath('data_holdout/patch-9_mask.tif'),
 'patch-10': PosixPath('data_holdout/patch-10_mask.tif'),
 'patch-6': PosixPath('data_holdout/patch-6_mask.tif')}

In [ ]:
# check that every ground-truth mask exists
for p, gt in gt_masks.items():
    assert gt.exists(), f"Missing GT for {p}"

In [ ]:
# bootstrap_models0 / 1 / 2 are the round 0 / 4 / 9 models
# collect every reconstruction (one per stack and model pair)
runs = []

for patch_dir in RESULTS_ROOT.iterdir():
    if not patch_dir.is_dir():
        continue

    patch_name = patch_dir.name
    if patch_name not in gt_masks:
        print(f"Skipping {patch_name} (no GT)")
        continue

    for mask_path in patch_dir.glob("*_mask.tif"):
        meta_path = mask_path.with_name(
            mask_path.name.replace("_mask.tif", "_meta.json")
        )

        runs.append({
            "patch": patch_name,
            "model_id": mask_path.stem.replace("_mask", ""),
            "mask_path": mask_path,
            "meta_path": meta_path if meta_path.exists() else None,
        })

print(f"Found {len(runs)} (patch × model) reconstructions")


In [ ]:
# load in the gt data
gt_data = {
    patch: tif.imread(path)
    for patch, path in gt_masks.items()
}

In [ ]:
from benchmarking_scripts.reconstruction_metrics import compute_AP

all_metrics = []

for run in tqdm(runs, desc="Evaluating reconstructions"):
    patch = run["patch"]
    model_id = run["model_id"]

    pred = tif.imread(run["mask_path"])
    gt = gt_data[patch]

    assert pred.shape == gt.shape, (
        f"Shape mismatch for {patch} / {model_id}: "
        f"{pred.shape} vs {gt.shape}"
    )

    # ---- AP computation (single sample) ----
    ap, tp, fp, fn = compute_AP(
        reference_list=[gt],
        predictions_list=[pred],
        ap50_only=True,
    )

    # unwrap single-element lists
    metrics = {
        "ap50": ap[0],
        "tp": tp[0],
        "fp": fp[0],
        "fn": fn[0],
    }

    # Attach identifiers
    metrics["patch"] = patch
    metrics["model_id"] = model_id

    # Attach metadata if available
    if run["meta_path"] is not None:
        with open(run["meta_path"]) as f:
            meta = json.load(f)
        metrics.update(meta)

    all_metrics.append(metrics)


In [ ]:
df = pd.DataFrame(all_metrics)
df.head()

In [ ]:
df.to_csv(OUT_DIR / "3DRecon_metrics_BootStrapped.csv")   # copy to 4_metrics/ for the plots